# AFS Analyze

Generates a consulting briefing for a loaded organization: ratio table, trend deltas, and `COMMON.FINDINGS`.

In [ ]:
import sys
sys.path.insert(0, '../src')

from afs.cortex_llm import init as llm_init
from afs.snowflake_io import cursor_from_session
from afs.insights import compute_ratios, compute_trends, synthesize_findings, write_findings

llm_init(session)
_ctx = cursor_from_session(session, commit_on_exit=False)
cur = _ctx.__enter__()

In [ ]:
# Show available organizations
import pandas as pd

orgs = session.sql("""
    SELECT ORG_CODE, LEGAL_NAME, SECTOR, HQ_STATE
      FROM AUDITED_FINANCIALS.COMMON.ORG_REGISTRY
     ORDER BY LEGAL_NAME
""").to_pandas()

print(f'{len(orgs)} organization(s) in registry:')
orgs

In [ ]:
ORG_CODE = 'CHESAPEAKE_HOSPITAL_AUTHORITY'

cur.execute(
    "SELECT ORG_ID, LEGAL_NAME FROM AUDITED_FINANCIALS.COMMON.ORG_REGISTRY WHERE ORG_CODE = %s",
    (ORG_CODE,)
)
row = cur.fetchone()
if not row:
    raise ValueError(f'ORG_CODE not found: {ORG_CODE}')
org_id, legal_name = row
print(f'Organization: {legal_name} ({ORG_CODE})')

In [ ]:
# Compute ratios
ratios = compute_ratios(cur, org_id)
trends = compute_trends(ratios)

# Display ratio table
years = ratios['years']
ratio_rows = []
for metric in next(iter(ratios['ratios_by_fy'].values()), {}):
    row_data = {'metric': metric}
    for yr in years:
        v = ratios['ratios_by_fy'].get(yr, {}).get(metric)
        row_data[yr] = f'{v:,.2f}' if v is not None else '—'
    ratio_rows.append(row_data)

df_ratios = pd.DataFrame(ratio_rows).set_index('metric')
print(f'\nRatios ({legal_name}):')
df_ratios

In [ ]:
findings = synthesize_findings(ratios, trends, ORG_CODE)
print(f'Generated {len(findings)} finding(s).')

cur.execute("""
    SELECT FILING_ID, FY_LABEL FROM AUDITED_FINANCIALS.COMMON.FILINGS
     WHERE ORG_ID = %s ORDER BY FISCAL_YEAR_END DESC LIMIT 1
""", (org_id,))
filing_row = cur.fetchone()
if filing_row:
    filing_id, fy_label = filing_row
    n = write_findings(cur, org_id, filing_id, fy_label, findings)
    cur.connection.commit()
    print(f'Wrote {n} finding(s) for {fy_label}.')

In [ ]:
from snowflake.snowpark.functions import col

df_findings = session.table('AUDITED_FINANCIALS.COMMON.FINDINGS').filter(
    col('ORG_ID') == org_id
).select(
    'FY_LABEL', 'SEVERITY', 'CATEGORY', 'TITLE',
    'EST_IMPACT_LOW', 'EST_IMPACT_HIGH', 'NARRATIVE', 'PLAYBOOK_HINT'
).order_by(col('CREATED_AT').desc()).limit(20).to_pandas()

for _, f in df_findings.iterrows():
    lo = f.get('EST_IMPACT_LOW')
    hi = f.get('EST_IMPACT_HIGH')
    rng = f'${lo:,.0f}\u2013${hi:,.0f}' if lo and hi else '\u2014'
    print(f"\n[{f['SEVERITY'].upper()}] {f['TITLE']}  ({f['CATEGORY']}, {rng})")
    print(f"  FY: {f['FY_LABEL']} | Playbook: {f['PLAYBOOK_HINT'] or '\u2014'}")
    print(f"  {f['NARRATIVE']}")


In [ ]:
_ctx.__exit__(None, None, None)

In [ ]:
from snowflake.snowpark.context import get_active_session
session = get_active_session()